# V2G capacity experiments

Set CPLEX_BIN to the installed IBM CPLEX Studio executable, then run all cells.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

project = Path.cwd()
data_archive = project / "study-data.zip"
if not (os.environ.get("CPLEX_BIN") or os.environ.get("CPLEX_LIBRARY")):
    raise RuntimeError("Set CPLEX_BIN or CPLEX_LIBRARY to your CPLEX Studio installation.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(project / "requirements-capacity.txt")], check=True)

## Prepare a fresh run

The archive contains inputs and protocols. Each run writes its own results.

In [ ]:
import shutil
import tempfile
from zipfile import ZipFile

runs = project / "runs"
runs.mkdir(exist_ok=True)
work = Path(tempfile.mkdtemp(dir=runs))
for script in (project / "replication").rglob("*.py"):
    target = work / script.relative_to(project)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(script, target)
with ZipFile(data_archive) as archive:
    for item in archive.infolist():
        path = Path(item.filename)
        if path.is_absolute() or ".." in path.parts or path.parts[0] != "replication" or path.suffix not in (".csv", ".json"):
            raise ValueError(f"Unexpected data file: {path}")
        archive.extract(item, work)
(work / "outputs").mkdir()
print(work)

## Solve with CPLEX and build the figures

In [ ]:
subprocess.run([sys.executable, "-u", "replication/run_focused_study.py"], cwd=work, check=True)

## Results

In [ ]:
import json

results = work / "replication/independent_validation/results"
print(json.loads((results / "validation_summary.json").read_text())["audit"])
print("Figures and tables:", work / "outputs")